# Experiment 3: Neural Network on MNIST

**Name:** Noora  
**Roll No:** 2301420015  
**Course:** B.Tech CSE (Data Science)

---

## Objective
To build, train, and evaluate a fully-connected neural network for handwritten digit classification using the MNIST dataset.

## Theory
A **feedforward neural network** consists of:
- **Input Layer**: Accepts flattened image (784 pixels)
- **Hidden Layer**: Learns feature representations using ReLU activation
- **Output Layer**: Produces class probabilities using Softmax (10 classes: 0-9)

Training uses **Adam optimizer** with **Sparse Categorical Cross-Entropy** loss.

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

sns.set(style='whitegrid', font_scale=1.1)
tf.random.set_seed(42)

# --- Load & Preprocess Data ---
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train / 255.0
x_test  = x_test  / 255.0

print(f"Training samples: {x_train.shape[0]} | Test samples: {x_test.shape[0]}")
print(f"Image shape: {x_train.shape[1:]}")

# --- Visualize Sample Images ---
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Sample MNIST Digits', fontsize=14, fontweight='bold')
for i, ax in enumerate(axes.flatten()):
    ax.imshow(x_train[i], cmap='gray')
    ax.set_title(f'Label: {y_train[i]}')
    ax.axis('off')
plt.tight_layout()
plt.savefig('exp3_samples.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Build Model ---
model = keras.Sequential([
    keras.layers.Flatten(input_shape=(28, 28)),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(10, activation='softmax')
], name='MNIST_Classifier')

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# --- Train ---
history = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.15,
    verbose=1
)

# --- Evaluate ---
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nTest Accuracy: {test_acc*100:.2f}%")
print(f"Test Loss    : {test_loss:.4f}")

In [ ]:
# --- Plot Training History ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Neural Network Training History', fontsize=14, fontweight='bold')

axes[0].plot(history.history['accuracy'], label='Train Accuracy', color='blue')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', color='orange')
axes[0].set_title('Accuracy over Epochs')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train Loss', color='blue')
axes[1].plot(history.history['val_loss'], label='Val Loss', color='orange')
axes[1].set_title('Loss over Epochs')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig('exp3_training.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Confusion Matrix ---
y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.title('Confusion Matrix - MNIST Classifier', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label'); plt.ylabel('True Label')
plt.tight_layout()
plt.savefig('exp3_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## Result
- The model achieves ~97-98% accuracy on the MNIST test set.
- Dropout layers help prevent overfitting (train vs val accuracy close).
- The confusion matrix shows most misclassifications are between visually similar digits (4/9, 3/5).

## Conclusion
A simple feedforward neural network with 2 hidden layers achieves high accuracy on MNIST. Dropout regularization improves generalization. This model architecture is the backbone for building more complex generative models like GANs and VAEs.